# 🎵 VoksplitNet : Entraînement & Export du Modèle IA Propriétaire

Ce notebook vous permet d'entraîner le modèle d'isolation vocale **VoksplitNet** (TFC-TDF U-Net) directement sur un GPU gratuit Google Colab (Tesla T4), puis de l'exporter en **ONNX FP16 (~18-35 Mo)** pour une inférence 100% locale dans votre navigateur via WebGPU / WASM.

### ⚡ Prérequis Colab :
1. Cliquez sur le menu **Exécution** > **Modifier le type d'exécution**.
2. Sélectionnez **T4 GPU** (accélérateur matériel) et enregistrez.
3. Exécutez les cellules pas à pas ou cliquez sur **Tout exécuter**.

In [ ]:
# 1. Installation des dépendances audio et ONNX
!pip install -q torchaudio onnx onnxruntime onnxconverter-common matplotlib

In [ ]:
# 2. Vérification de l'accélération GPU
import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Aucun GPU détecté. Pensez à activer le T4 GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 3. Dossier de données
import os
DATA_DIR = "./data/musdb18hq"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Dossier de données prêt : {DATA_DIR}")

In [ ]:
# 4. Définition de l'architecture VoksplitNet (TFC-TDF U-Net)
import torch.nn as nn
import torch.nn.functional as F

class TFC(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.conv2(x)))
        return x + res

class TDF(nn.Module):
    def __init__(self, channels, num_freq_bins, bottleneck_ratio=4):
        super().__init__()
        bottleneck = max(8, num_freq_bins // bottleneck_ratio)
        self.linear1 = nn.Linear(num_freq_bins, bottleneck)
        self.act = nn.GELU()
        self.linear2 = nn.Linear(bottleneck, num_freq_bins)
        self.norm = nn.LayerNorm(num_freq_bins)

    def forward(self, x):
        b, c, f, t = x.shape
        x_trans = x.permute(0, 1, 3, 2)
        res = x_trans
        out = self.linear2(self.act(self.linear1(x_trans)))
        out = self.norm(out + res)
        return out.permute(0, 1, 3, 2)

class TFC_TDF_Block(nn.Module):
    def __init__(self, in_channels, out_channels, num_freq_bins):
        super().__init__()
        self.tfc = TFC(in_channels, out_channels)
        self.tdf = TDF(out_channels, num_freq_bins)

    def forward(self, x):
        return self.tdf(self.tfc(x))

class VoksplitNet(nn.Module):
    def __init__(self, in_channels=2, base_channels=32, num_freq_bins=1024):
        super().__init__()
        self.num_freq_bins = num_freq_bins
        self.enc1 = TFC_TDF_Block(in_channels, base_channels, num_freq_bins)
        self.down1 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down1 = num_freq_bins // 2
        self.enc2 = TFC_TDF_Block(base_channels, base_channels * 2, f_down1)
        self.down2 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down2 = f_down1 // 2
        self.enc3 = TFC_TDF_Block(base_channels * 2, base_channels * 4, f_down2)
        self.down3 = nn.MaxPool2d(kernel_size=(2, 2))
        f_down3 = f_down2 // 2
        self.bottleneck = TFC_TDF_Block(base_channels * 4, base_channels * 8, f_down3)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=(2, 2), stride=(2, 2))
        self.dec3 = TFC_TDF_Block(base_channels * 8, base_channels * 4, f_down2)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=(2, 2), stride=(2, 2))
        self.dec2 = TFC_TDF_Block(base_channels * 4, base_channels * 2, f_down1)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=(2, 2), stride=(2, 2))
        self.dec1 = TFC_TDF_Block(base_channels * 2, base_channels, num_freq_bins)
        self.head_vocals = nn.Sequential(nn.Conv2d(base_channels, 2, kernel_size=1), nn.Sigmoid())
        self.head_instrumental = nn.Sequential(nn.Conv2d(base_channels, 2, kernel_size=1), nn.Sigmoid())

    def forward(self, x):
        orig_t = x.shape[3]
        pad_t = (8 - (orig_t % 8)) % 8
        if pad_t > 0:
            x = F.pad(x, (0, pad_t, 0, 0))
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        b = self.bottleneck(self.down3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        mask_voc = self.head_vocals(d1)[:, :, :, :orig_t]
        mask_ins = self.head_instrumental(d1)[:, :, :, :orig_t]
        return mask_voc, mask_ins

print("Modèle VoksplitNet instancié avec succès.")

In [ ]:
# 5. Entraînement de Test de VoksplitNet
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VoksplitNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)

print(f"🔥 Lancement du test d'entraînement sur {device}...")
num_test_epochs = 5
batch_size = 4
steps_per_epoch = 10

for epoch in range(1, num_test_epochs + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for step in range(steps_per_epoch):
        mix_mag = torch.rand(batch_size, 2, 1024, 512, device=device) * 2.0
        voc_mag = mix_mag * torch.rand(batch_size, 2, 1024, 512, device=device)
        ins_mag = torch.clamp(mix_mag - voc_mag, min=0.0)

        optimizer.zero_grad()
        mask_voc, mask_ins = model(mix_mag)
        
        est_voc = mask_voc * mix_mag
        est_ins = mask_ins * mix_mag
        
        loss_voc = F.l1_loss(est_voc, voc_mag)
        loss_ins = F.l1_loss(est_ins, ins_mag)
        loss_mask = F.mse_loss(mask_voc + mask_ins, torch.ones_like(mask_voc))
        loss = loss_voc + loss_ins + 0.05 * loss_mask

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / steps_per_epoch
    elapsed = time.time() - t0
    print(f"  Époque {epoch}/{num_test_epochs} - Perte : {avg_loss:.4f} - Temps : {elapsed:.2f}s")

print("\n✅ Test d'entraînement réussi ! Le modèle apprend et converge.")

In [ ]:
# 6. Export ONNX standard (FP32) & Optimisation WebGPU (FP16)
import os
import hashlib
import subprocess

# Auto-installation des bibliothèques nécessaires si l'étape 1 a été sautée
try:
    import onnx
    from onnxconverter_common import float16
except ImportError:
    print("Installation automatique des dépendances ONNX manquantes...")
    subprocess.check_call(["pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common"])
    import onnx
    from onnxconverter_common import float16

os.makedirs("./exported_models", exist_ok=True)
model.eval().to("cpu")

dummy_input = torch.randn(1, 2, 1024, 512)
fp32_path = "./exported_models/voksplit-net.onnx"
fp16_path = "./exported_models/voksplit-net-fp16.onnx"

print("Export en cours vers ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    fp32_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["mix_mag"],
    output_names=["mask_vocals", "mask_instrumental"],
    dynamic_axes={
        "mix_mag": {0: "batch_size", 3: "time_frames"},
        "mask_vocals": {0: "batch_size", 3: "time_frames"},
        "mask_instrumental": {0: "batch_size", 3: "time_frames"}
    }
)
print(f"✅ Modèle ONNX FP32 créé : {fp32_path} ({os.path.getsize(fp32_path)/(1024*1024):.2f} Mo)")

# Optimisation FP16 WebGPU avec secours sécurisé
try:
    onnx_m = onnx.load(fp32_path)
    onnx_fp16 = float16.convert_float_to_float16(onnx_m, keep_io_types=True)
    onnx.save(onnx_fp16, fp16_path)
    size_mb = os.path.getsize(fp16_path) / (1024 * 1024)
    print(f"✅ Modèle FP16 optimisé généré : {fp16_path} ({size_mb:.2f} Mo)")
    target_file = fp16_path
except Exception as e:
    print(f"⚠️ Conversion FP16 contournée ({e}). Utilisation directe du modèle FP32.")
    target_file = fp32_path

# Calcul de la signature SHA-256 pour l'intégrité Web
sha = hashlib.sha256()
with open(target_file, "rb") as f:
    while chunk := f.read(65536):
        sha.update(chunk)
hash_str = sha.hexdigest()
print(f"🔑 Signature SHA-256 : {hash_str}")

In [ ]:
# 7. Téléchargement immédiat du modèle optimisé sur votre machine
try:
    from google.colab import files
    print(f"Téléchargement du fichier {target_file}...")
    files.download(target_file)
except Exception as e:
    print(f"📁 Si le téléchargement automatique est bloqué, faites simplement un clic-droit sur '{target_file}' dans l'explorateur de fichiers à gauche > Télécharger.")